# Daily Appointment Data Generator

## Goal

Use `vcu_master_daily.csv` to create synthetic appointment/event records for every admission counted on every day in the dataset. First inspect admission counts by category, then expand those counts into individual records.

Each generated record has a unique ID, the source date and admission category, and a randomly assigned check-in time within the 24-hour day. The source key links each record back to its daily row in `vcu_master_daily.csv`.

## Data limitations

`vcu_master_daily.csv` contains aggregate daily counts, not observed appointment-level records or check-in times. The generated records and times are therefore synthetic. Randomizing times across the full 24 hours is an explicit placeholder assumption; these times should not be interpreted as actual hospital scheduling patterns.

In [1]:
from pathlib import Path
import pandas as pd

csv_path = Path("data") / "vcu_master_daily.csv"
df = pd.read_csv(csv_path)

df.head()

,date,total_staffed_beds,total_occupied_beds,total_free_beds,occupied_beds_newborn,occupied_beds_delivery,occupied_beds_non_elective_via_ed,occupied_beds_non_elective_other,occupied_beds_elective,occupied_beds_covid_19,...,expected_discharges_other_viral,expected_discharges_intensive,avg_los_newborn_days,avg_los_delivery_days,avg_los_non_elective_via_ed_days,avg_los_non_elective_other_days,avg_los_elective_days,avg_los_covid_19_days,avg_los_influenza_days,avg_los_other_viral_days
0,2017-01-01,905,660.0,245.0,43,47,100,120,76,NaN,...,18.898079,16.090096,2.096227,2.604650,5.144324,6.663770,4.170857,NaN,4.591475,5.386618
1,2017-01-02,905,659.0,246.0,38,45,104,114,83,NaN,...,18.981292,15.397648,2.081085,2.523753,5.118284,6.379377,4.126970,NaN,4.651885,5.368373
2,2017-01-03,905,653.0,252.0,33,41,104,117,84,NaN,...,18.971664,15.838223,2.120760,2.518487,5.044225,6.367644,4.159663,NaN,4.680748,5.582925
3,2017-01-04,905,642.0,263.0,29,38,108,114,73,NaN,...,19.084184,15.696313,2.086112,2.588535,4.968106,6.465672,4.217788,NaN,4.643710,5.499093
4,2017-01-05,905,652.0,253.0,29,34,103,121,75,NaN,...,17.747718,16.763519,2.124785,2.665197,4.989300,6.521617,4.230880,NaN,4.624938,5.570379


In [ ]:
df["key"] = range(1, len(df) + 1)
df.head()

,date,total_staffed_beds,total_occupied_beds,total_free_beds,occupied_beds_newborn,occupied_beds_delivery,occupied_beds_non_elective_via_ed,occupied_beds_non_elective_other,occupied_beds_elective,occupied_beds_covid_19,...,expected_discharges_intensive,avg_los_newborn_days,avg_los_delivery_days,avg_los_non_elective_via_ed_days,avg_los_non_elective_other_days,avg_los_elective_days,avg_los_covid_19_days,avg_los_influenza_days,avg_los_other_viral_days,key
0,2017-01-01,905,660.0,245.0,43,47,100,120,76,NaN,...,16.090096,2.096227,2.604650,5.144324,6.663770,4.170857,NaN,4.591475,5.386618,1
1,2017-01-02,905,659.0,246.0,38,45,104,114,83,NaN,...,15.397648,2.081085,2.523753,5.118284,6.379377,4.126970,NaN,4.651885,5.368373,2
2,2017-01-03,905,653.0,252.0,33,41,104,117,84,NaN,...,15.838223,2.120760,2.518487,5.044225,6.367644,4.159663,NaN,4.680748,5.582925,3
3,2017-01-04,905,642.0,263.0,29,38,108,114,73,NaN,...,15.696313,2.086112,2.588535,4.968106,6.465672,4.217788,NaN,4.643710,5.499093,4
4,2017-01-05,905,652.0,253.0,29,34,103,121,75,NaN,...,16.763519,2.124785,2.665197,4.989300,6.521617,4.230880,NaN,4.624938,5.570379,5


In [4]:
df.loc[df["key"] == 1]


,date,total_staffed_beds,total_occupied_beds,total_free_beds,occupied_beds_newborn,occupied_beds_delivery,occupied_beds_non_elective_via_ed,occupied_beds_non_elective_other,occupied_beds_elective,occupied_beds_covid_19,...,expected_discharges_intensive,avg_los_newborn_days,avg_los_delivery_days,avg_los_non_elective_via_ed_days,avg_los_non_elective_other_days,avg_los_elective_days,avg_los_covid_19_days,avg_los_influenza_days,avg_los_other_viral_days,key
0,2017-01-01,905,660.0,245.0,43,47,100,120,76,NaN,...,16.090096,2.096227,2.60465,5.144324,6.66377,4.170857,NaN,4.591475,5.386618,1


In [8]:
import numpy as np

from datetime import timedelta



# ASSUMPTION: Each event time is sampled uniformly across the full 24-hour day.

rng = np.random.default_rng(seed=2026)



admission_columns = [column for column in df.columns if column.startswith("admissions_")]

appointment_records = []



for _, source_row in df.iterrows():

    for column in admission_columns:

        category = column.removeprefix("admissions_")

        count = int(source_row[column]) if pd.notna(source_row[column]) else 0

        for category_event_number in range(1, count + 1):

            seconds_after_midnight = int(rng.integers(0, 24 * 60 * 60))

            appointment_records.append({

                "appointment_id": f"key-{int(source_row['key'])}-{category}-{category_event_number:04d}",

                "source_key": int(source_row["key"]),

                "date": source_row["date"],

                "category": category,

                "check_in_time": (pd.Timestamp(source_row["date"]) + timedelta(seconds=seconds_after_midnight)).time(),

            })



appointments = pd.DataFrame(appointment_records)

appointments = appointments.sort_values(["date", "check_in_time"]).reset_index(drop=True)

expected_count = int(df[admission_columns].fillna(0).to_numpy().sum())

assert len(appointments) == expected_count

assert appointments["appointment_id"].is_unique



print(f"Daily source rows: {len(df):,}")

print(f"Generated appointment records: {len(appointments):,}")

display(appointments["category"].value_counts().rename_axis("category").reset_index(name="appointment_count"))

appointments.head(10)

Daily source rows: 2,922
Generated appointment records: 372,197


,category,appointment_count
0,non_elective_via_ed,43800
1,non_elective_other,43742
2,newborn,43638
3,influenza,43577
4,delivery,43554
5,elective,43519
6,other_viral,43492
7,intensive,41674
8,covid_19,25201


,appointment_id,source_key,date,category,check_in_time
0,key-1-influenza-0014,1,2017-01-01,influenza,00:02:35
1,key-1-non_elective_other-0012,1,2017-01-01,non_elective_other,00:05:58
2,key-1-non_elective_via_ed-0008,1,2017-01-01,non_elective_via_ed,00:18:29
3,key-1-newborn-0003,1,2017-01-01,newborn,00:38:02
4,key-1-influenza-0007,1,2017-01-01,influenza,00:48:56
5,key-1-elective-0018,1,2017-01-01,elective,00:49:53
6,key-1-influenza-0015,1,2017-01-01,influenza,00:56:07
7,key-1-other_viral-0015,1,2017-01-01,other_viral,01:24:34
8,key-1-non_elective_other-0018,1,2017-01-01,non_elective_other,01:52:49
9,key-1-newborn-0007,1,2017-01-01,newborn,01:54:57


In [10]:
output_path = Path("data") / "synthetic_appointments.csv"
appointments.to_csv(output_path, index=False)
print(f"Saved {len(appointments):,} appointment records to {output_path}")

Saved 372,197 appointment records to data\synthetic_appointments.csv
